# BibTeX Reference Verifier

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aliakarma/academic-agent-skills/blob/main/notebook/BibTeX_DOI_Verifier.ipynb)

Checks **every field** of **every entry** in a `.bib` file against public bibliographic registries and
reports what is wrong, missing, or disputed. Works for any field: computer science, biomedicine, physics,
engineering, social science, and books.

This notebook is a front end for
[`skills/reference-verify/scripts/verify_bib.py`](../skills/reference-verify/scripts/verify_bib.py), the
whole-file mode of the [`reference-verify`](../skills/reference-verify/SKILL.md) skill. All the logic lives
in that script, so the notebook, the command line, and AI agents give the same results.

## What is checked

| Field | Checked against |
|---|---|
| `doi` | CrossRef, DataCite, OpenAlex, and DOI content negotiation (any registrar) |
| `title` | registry title, ignoring LaTeX, accents and case |
| `author` | every author: identity, order, count, given-name form |
| `journal` / `booktitle` | container title, with acronym and abbreviation matching |
| `year` | published-version year (preprint records excluded) |
| `volume`, `number`, `pages` | publisher of record first, then registries |
| `publisher`, `issn`, `isbn`, `edition` | registry metadata |
| `url` | live HTTP fetch, with retries; bot blocks (202, 403, 418) are not called dead |
| retractions | CrossRef `updated-by` notices (Retraction Watch data) and OpenAlex |
| duplicates | the same DOI or title under two citation keys |
| entry type | registry work type vs BibTeX `@type` |

Sources, in order of authority: publisher pages that serve their own BibTeX (PMLR, ACL Anthology),
CrossRef, Europe PMC, DBLP, OpenAlex, DataCite, OpenLibrary, DOI content negotiation, arXiv.

## Two rules that keep it from corrupting a good `.bib`

1. **Only identity-confirmed records may donate a value**: a record matched by DOI, by the publisher of
   record, by ISBN, or by a near-identical distinctive title. A merely similar paper never lends its volume
   or page range to yours.
2. **Disagreement is reported, never resolved.** When two trusted registries give different values, or one
   record disagrees with your whole author list, the field becomes `REVIEW` and is left untouched.

By default nothing is written: you get a report. With `APPLY = True` only the corrected values change in
your file; field order, `@string` macros and your comments stay as they are, and the original is kept as
`<file>.orig.bak`.

> A resolving DOI proves existence, not correctness. Every `REVIEW` row still needs your eye.

**Privacy.** Titles, DOIs, ISBNs, arXiv IDs and first-author surnames are sent to the public APIs above.
Your contact e-mail, if you set one, goes in the request header so the services can reach you about
heavy use.

## 1. Install and fetch the verifier

Uses the copy in this repository when the notebook runs from a clone, and downloads it otherwise (Colab).

In [ ]:
import os, subprocess, sys, urllib.request

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'bibtexparser>=1.4,<2', 'requests>=2.28,<3'], check=True)

LOCAL = os.path.join('..', 'skills', 'reference-verify', 'scripts', 'verify_bib.py')
RAW = ('https://raw.githubusercontent.com/aliakarma/academic-agent-skills/main/'
       'skills/reference-verify/scripts/verify_bib.py')
SCRIPT = LOCAL if os.path.exists(LOCAL) else 'verify_bib.py'
if not os.path.exists(SCRIPT):
    urllib.request.urlretrieve(RAW, SCRIPT)
print('Using', os.path.abspath(SCRIPT))

## 2. Settings

Optional: put a `venue_aliases.json` file next to your `.bib` with extra venue synonym groups, for example
`[["my workshop on x", "mwx"], ["journal of foo", "j. foo"]]`.

In [ ]:
BIB_PATH      = 'references.bib'  # in Colab, leave as is and upload the file in step 3
CONTACT_EMAIL = ''                # e.g. name@example.com, for the CrossRef/OpenAlex polite pools
APPLY         = False             # True = write corrections into the .bib (a .orig.bak backup is kept)
CHECK_URLS    = True              # live-fetch every url field
WORKERS       = 6                 # entries checked in parallel; per-host rate limits still apply

## 3. Upload the `.bib` (Colab only)

In [ ]:
if 'google.colab' in sys.modules and not os.path.exists(BIB_PATH):
    from google.colab import files
    BIB_PATH = next(iter(files.upload()))
print('Checking', BIB_PATH)

## 4. Run

Progress is printed as each entry finishes. The same run from a terminal:

```
python skills/reference-verify/scripts/verify_bib.py references.bib --email name@example.com
python skills/reference-verify/scripts/verify_bib.py references.bib --apply
```

In [ ]:
cmd = [sys.executable, SCRIPT, BIB_PATH, '--workers', str(WORKERS)]
cmd += ['--email', CONTACT_EMAIL] if CONTACT_EMAIL else []
cmd += ['--apply'] if APPLY else []
cmd += [] if CHECK_URLS else ['--no-urls']

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, encoding='utf-8')
for line in proc.stdout:
    print(line, end='')
status = proc.wait()
print(f'\nExit status {status}: '
      + {0: 'every entry checked out.', 1: 'some entries need a human look - see the report below.'}
      .get(status, 'the run failed.'))

## 5. Read the report

| Entry status | Meaning |
|---|---|
| `CLEAN` | every checked field matches |
| `NORMALIZED` | only formatting was fixed (DOI prefix, page dashes) |
| `REVIEW` | something could not be confirmed or sources disagree |
| `PROBLEM` | a field is wrong, the DOI points to another work, or the work is retracted |
| `UNVERIFIED` | no registry has this record: check it by hand, it may be fabricated |

For each `REVIEW` or `PROBLEM` entry, the next step is the `reference-verify` skill: give your agent the
entry and the report row, and it checks the publisher page and returns a corrected BibTeX entry.

In [ ]:
from IPython.display import Markdown, display

with open('verification_report.md', encoding='utf-8') as fh:
    display(Markdown(fh.read()))

In [ ]:
# Colab: download the report (and the corrected .bib with its backup, if APPLY was on)
if 'google.colab' in sys.modules:
    from google.colab import files
    outputs = ['verification_report.md', 'verification_report.csv']
    if APPLY:
        outputs += [BIB_PATH, BIB_PATH + '.orig.bak']
    for f in outputs:
        if os.path.exists(f):
            files.download(f)